## RAGAS Evaluation Setup

In most RAG projects evaluation is an afterthought bolted on at the end, which
means teams optimise for what they can measure easily rather than for what
actually matters. This notebook reverses that order: before you ingest a single
document or build a retrieval chain for CatalogueIQ, you define *what good looks
like* — a golden test set of 10+ question-answer pairs, one evaluation harness,
and four RAGAS metrics that will be your north star through Day 1, Day 2, and
the final demo. You will also run RAGAS against dummy (poor-quality) answers so
you can see what low scores look like and calibrate your intuitions.

### Concepts Covered

- The four RAGAS metrics and what each one catches in a ShopSmart India context
- Why **faithfulness** matters most for product spec queries (hallucinated specs are a serious trust problem)
- Why **context recall** matters most for multi-hop queries (missing a policy section = wrong answer)
- Building a golden test set that covers all five CatalogueIQ query types
- Writing ShopSmart India domain-specific ground-truth answers with Indian pricing (₹) and real product names
- Running RAGAS against deliberately bad answers to see low baseline scores
- How the golden test set connects to NB-08 final evaluation

In [1]:
import ragas
import langchain
import langchain_community

print(ragas.__version__)
print(langchain.__version__)
print(langchain_community.__version__)

/Users/nareshchaurasia/nc/PYTHON-ARCHITECT/Python-Immersive-AI/week2-mac/.venv_rag/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


0.2.15
0.3.26
0.3.27


In [2]:
# ── SETUP ──────────────────────────────────────────────────────
import os
import json
from dotenv import load_dotenv

load_dotenv()  # reads ANTHROPIC_API_KEY from .env

MODEL = "claude-sonnet-4-6"
EMBEDDING_MODEL = "all-MiniLM-L6-v2"

# RAGAS imports
from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall,
)
from datasets import Dataset

print("RAGAS evaluation environment ready.")
print(f"LLM model for RAGAS judge: {MODEL}")


RAGAS evaluation environment ready.
LLM model for RAGAS judge: claude-sonnet-4-6


### Step 1 — The Four RAGAS Metrics Explained

Understanding what each metric measures in the CatalogueIQ context:

| Metric | What it measures | Why it matters for ShopSmart India |
|---|---|---|
| **Faithfulness** | Does the answer contain only facts present in the retrieved context? | Critical for product specs — the model must never invent a battery life figure |
| **Answer Relevancy** | Does the answer actually address the question asked? | A shopper asking about return policy should not get a product recommendation |
| **Context Precision** | Are the retrieved chunks ranked with the most relevant ones first? | Comparative queries need the right products at the top, not buried at rank 5 |
| **Context Recall** | Does the retrieved context contain all information needed for the ground-truth answer? | Multi-hop queries need chunks from at least 3 documents; missing any = incomplete answer |

**Target scores for CatalogueIQ after improvement:**
- Faithfulness ≥ 0.85 (non-negotiable — hallucinated specs damage trust)
- Answer Relevancy ≥ 0.80
- Context Precision ≥ 0.75
- Context Recall ≥ 0.70 (hardest to achieve on multi-hop queries)

In [3]:
# ── GOLDEN TEST SET ───────────────────────────────────────────
# 10 question-answer pairs covering all 5 CatalogueIQ query types.
# These are GROUND TRUTH — they were written before any pipeline was built.
# The RAGAS evaluation in Notebook-008 will compare generated answers against these.

# COST NOTE: No API calls in this cell. The test set is hard-coded.

golden_test_set = [
    # ── Query Type 1: Product Factual Lookup (2 questions) ──────
    {
        "question": "What is the total battery life of the boAt Airdopes 141 earbuds?",
        "ground_truth": (
            "The boAt Airdopes 141 earbuds have a total battery life of 42 hours — "
            "up to 6 hours in the earbuds themselves plus an additional 36 hours "
            "from the charging case. They support IPX4 water resistance and "
            "Bluetooth 5.1 connectivity."
        ),
        "query_type": "product_factual",
    },
    {
        "question": "What are the connectivity specs and driver size of the Jabra Evolve2 Buds?",
        "ground_truth": (
            "The Jabra Evolve2 Buds use Bluetooth 5.2 and have a 6mm driver size. "
            "They carry an IP57 water and dust resistance rating and provide "
            "up to 33 hours of total battery life including the charging case."
        ),
        "query_type": "product_factual",
    },

    # ── Query Type 2: Policy & Eligibility (2 questions) ────────
    {
        "question": (
            "Can I return a fashion item I bought during the Big Billion Days sale "
            "if it does not fit?"
        ),
        "ground_truth": (
            "Yes. Fashion items purchased during the Big Billion Days sale on "
            "ShopSmart India are eligible for return within the standard 30-day "
            "return window, provided the item is unworn, has original tags attached, "
            "and is returned in its original packaging. The exception is items "
            "explicitly marked 'Final Sale' on the product page, which are not "
            "eligible for return or exchange under any circumstance."
        ),
        "query_type": "policy_eligibility",
    },
    {
        "question": "What is the return window for Electronics on ShopSmart India?",
        "ground_truth": (
            "Electronics on ShopSmart India have a 7-day return window from the "
            "date of delivery. The item must be returned in its original condition "
            "with all original packaging, accessories, and the purchase receipt."
        ),
        "query_type": "policy_eligibility",
    },

    # ── Query Type 3: Comparative Recommendation (2 questions) ──
    {
        "question": (
            "Which wireless earbuds under ₹3000 on ShopSmart India have "
            "noise cancellation and at least 20 hours total battery life?"
        ),
        "ground_truth": (
            "Among earbuds under ₹3000 on ShopSmart India, the Sony WF-C500 "
            "(₹2990) meets the battery criterion with 20 hours total battery life "
            "but does not support Active Noise Cancellation (ANC). "
            "The boAt Airdopes 141 (₹1299) offers 42 hours total battery but "
            "also lacks ANC. For ANC with extended battery life you would need to "
            "consider the Jabra Evolve2 Buds at ₹8999, which is above the ₹3000 budget."
        ),
        "query_type": "comparative_recommendation",
    },
    {
        "question": (
            "Compare the boAt Airdopes 141 and Sony WF-C500 — which offers "
            "better value for money for daily commute use?"
        ),
        "ground_truth": (
            "For daily commute use, the boAt Airdopes 141 (₹1299, rated 4.1/5) "
            "offers significantly longer battery life at 42 hours total compared to "
            "the Sony WF-C500 (₹2990, rated 4.3/5) at 20 hours. The Sony has a "
            "higher rating and slightly better sound quality per reviewers, but at "
            "more than double the price. For commuters prioritising battery life and "
            "budget, the boAt Airdopes 141 offers better value; for those prioritising "
            "sound quality and brand reputation, the Sony WF-C500 is the stronger choice."
        ),
        "query_type": "comparative_recommendation",
    },

    # ── Query Type 4: Seller Policy Lookup (2 questions) ────────
    {
        "question": (
            "What are the image resolution and background requirements for listing "
            "a product in the Electronics category on ShopSmart India?"
        ),
        "ground_truth": (
            "For Electronics listings on ShopSmart India, the hero image must be at "
            "least 1000 × 1000 pixels with a pure white background (RGB 255,255,255). "
            "Sellers must provide a minimum of 4 images and a maximum of 12, including "
            "front, back, side profile, and an in-use or lifestyle shot. Watermarks, "
            "seller logos, and text overlays are prohibited on the hero image. "
            "Files must be JPEG or PNG and must not exceed 5 MB per image."
        ),
        "query_type": "seller_policy",
    },
    {
        "question": "What documents must a new seller upload before their first listing goes live on ShopSmart India?",
        "ground_truth": (
            "Before their first listing goes live, a new seller on ShopSmart India must: "
            "upload and verify their GST registration number; link a bank account and "
            "complete penny-drop verification; upload a brand authorisation letter for "
            "branded products; ensure at least one product listing passes the automated "
            "image quality check; and set pricing at or below the printed MRP."
        ),
        "query_type": "seller_policy",
    },

    # ── Query Type 5: Multi-Hop Reasoning (2 questions) ─────────
    {
        "question": (
            "If I buy a smartwatch from a third-party seller on ShopSmart India "
            "and it stops working after 45 days, what are my options?"
        ),
        "ground_truth": (
            "After 45 days the standard 7-day Electronics return window has closed, "
            "so a direct return is not available. However, you have two remaining "
            "options: (1) Warranty claim — if the smartwatch has a manufacturer "
            "warranty (typically 12 or 24 months), you can contact the brand's "
            "authorised service centre for repair or replacement. "
            "(2) ShopSmart A-to-Z Guarantee — if the product stopped functioning "
            "due to a manufacturing defect and the seller disputes responsibility, "
            "you can raise an A-to-Z Guarantee claim within 30 days of the latest "
            "estimated delivery date. ShopSmart India will investigate and may issue "
            "a full refund even if the seller disputes it."
        ),
        "query_type": "multi_hop",
    },
    {
        "question": (
            "I placed a ₹4500 Electronics order on ShopSmart India using Cash on "
            "Delivery. Is that allowed, and if my order is delayed can I get a refund?"
        ),
        "ground_truth": (
            "No, a ₹4500 Electronics order cannot be placed using Cash on Delivery "
            "on ShopSmart India. COD is available only for orders up to ₹10,000 at "
            "eligible pin codes, but Electronics orders above ₹5,000 are not eligible "
            "for COD and require prepaid payment. For orders up to ₹5,000 COD may be "
            "available. If an eligible COD order is significantly delayed beyond the "
            "promised delivery date, you can raise an A-to-Z Guarantee claim for a "
            "full refund."
        ),
        "query_type": "multi_hop",
    },
]

print(f"Golden test set created: {len(golden_test_set)} questions")
for qt in ["product_factual", "policy_eligibility", "comparative_recommendation",
           "seller_policy", "multi_hop"]:
    count = sum(1 for q in golden_test_set if q["query_type"] == qt)
    print(f"  {qt}: {count} questions")


Golden test set created: 10 questions
  product_factual: 2 questions
  policy_eligibility: 2 questions
  comparative_recommendation: 2 questions
  seller_policy: 2 questions
  multi_hop: 2 questions


### Step 2 — Dummy Baseline to See Low RAGAS Scores

Before building the pipeline we run RAGAS with deliberately poor answers to
understand what low scores look like. This calibrates our expectations: a
faithfulness score of 0.15 means the answer is making up facts not in the
context; a context_recall of 0.20 means we are retrieving context that misses
most of what the ground truth requires.

**Note:** RAGAS calls the LLM judge (Claude) to evaluate each answer.
This step costs API tokens — a small run over 3 questions costs < $0.05.

In [ ]:
# COST NOTE: RAGAS calls the Claude API to judge each answer.
# This 3-question dummy run costs approximately $0.02–$0.05.
# To skip API calls, comment out the evaluate() call and use the printed
# placeholder scores instead.

from langchain_anthropic import ChatAnthropic
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_community.embeddings import HuggingFaceEmbeddings

# Set up RAGAS to use Claude as the judge LLM
llm = ChatAnthropic(model=MODEL, temperature=0)
ragas_llm = LangchainLLMWrapper(llm)

# Set up RAGAS to use SentenceTransformer for answer relevancy embedding
hf_embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)
ragas_embeddings = LangchainEmbeddingsWrapper(hf_embeddings)

# ── Build a 3-question dummy dataset with intentionally bad answers ──
dummy_data = {
    "question": [
        golden_test_set[0]["question"],   # product factual
        golden_test_set[2]["question"],   # policy eligibility
        golden_test_set[6]["question"],   # seller policy
    ],
    "answer": [
        # BAD answer 1: hallucinated spec
        "The boAt Airdopes 141 has a battery life of 100 hours and supports ANC.",
        # BAD answer 2: vague non-answer
        "You can return items according to the return policy.",
        # BAD answer 3: completely off-topic
        "ShopSmart India is a great place to shop online.",
    ],
    "contexts": [
        # Irrelevant context (simulating a retrieval failure)
        ["ShopSmart India sells products across Electronics, Fashion, and more."],
        ["ShopSmart India offers great deals during Big Billion Days."],
        ["Sellers must register on ShopSmart India to list products."],
    ],
    "ground_truth": [
        golden_test_set[0]["ground_truth"],
        golden_test_set[2]["ground_truth"],
        golden_test_set[6]["ground_truth"],
    ],
}

dummy_dataset = Dataset.from_dict(dummy_data)

print("Running RAGAS on 3 dummy (bad) answers to establish a low baseline...")
print("(This makes ~15-20 LLM API calls as RAGAS judges each metric)")

try:
    dummy_results = evaluate(
        dataset=dummy_dataset,
        metrics=[faithfulness, answer_relevancy, context_precision, context_recall],
        llm=ragas_llm,
        embeddings=ragas_embeddings,
    )
    print("\n── Dummy Baseline RAGAS Scores ─────────────────────────────")
    print(dummy_results)
    print("\n⚠️  Low scores expected: bad answers + irrelevant context = low metrics.")
    print("    In NB-08 we will rerun this with real pipeline outputs.")
except Exception as e:
    print(f"RAGAS evaluation error: {e}")
    print("\nExpected approximate scores for bad answers:")
    print("  faithfulness:      ~0.00–0.15  (answer contains hallucinated facts)")
    print("  answer_relevancy:  ~0.30–0.50  (vague answers still somewhat on-topic)")
    print("  context_precision: ~0.10–0.25  (wrong context retrieved)")
    print("  context_recall:    ~0.05–0.20  (context misses ground-truth information)")


In [ ]:
# ── SAVE GOLDEN TEST SET ──────────────────────────────────────
import os
os.makedirs("output", exist_ok=True)

output_path = "output/golden_test_set.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(golden_test_set, f, indent=2, ensure_ascii=False)

print(f"Golden test set saved to {output_path}")
print("This file will be loaded by NB-08 for the final RAGAS evaluation.")
print(f"Total questions: {len(golden_test_set)}")

# Print a summary table
print("\n── Golden Test Set Summary ─────────────────────────────────")
print(f"{'#':<4} {'Query Type':<28} {'Question (truncated)'}")
print("─" * 75)
for i, item in enumerate(golden_test_set, 1):
    q_short = item['question'][:45] + "…" if len(item['question']) > 45 else item['question']
    print(f"{i:<4} {item['query_type']:<28} {q_short}")


In [ ]:
# ── EXPERIMENT ────────────────────────────────────────────────

# EXERCISE 1 — Add more questions per query type
# The current test set has 2 questions per type. The capstone rubric expects
# at least 4 comparative recommendation questions (the hardest type).
# Add 2 more comparative recommendation questions below and append them to
# golden_test_set. Think about: "best smartwatch under ₹5000",
# "Sony vs boAt for call quality". What makes a good comparative ground truth?

# EXERCISE 2 — Experiment with context quality
# Replace the dummy_data contexts with PERFECT context (paste the actual
# policy text that answers each question). Re-run RAGAS.
# Which metrics go up? Which stay low? Why does faithfulness remain low
# if the answer still contains hallucinated information?

# EXERCISE 3 — Metric sensitivity
# Change one dummy answer from completely wrong to partially correct
# (e.g. correct battery life but wrong ANC claim).
# Re-run RAGAS. How sensitive is faithfulness to partial hallucination?
# This tells you the threshold at which the metric catches spec errors.

print("Exercises ready. Edit golden_test_set or dummy_data above and re-run.")


### Key Takeaway

Evaluation must come before construction. By defining the golden test set now,
you lock in what "good" means independent of what your system is currently good
at — preventing the common failure of tuning your pipeline to pass tests you
wrote after seeing the output. The four RAGAS metrics map directly onto the five
CatalogueIQ query types: faithfulness protects product fact accuracy,
context_recall exposes multi-hop retrieval gaps, and context_precision diagnoses
comparative recommendation failures.

In **NB-03** you will load and chunk the ShopSmart India knowledge base files —
starting with the design principle that a structured CSV row is its own natural
chunk, while unstructured policy prose needs sentence-level splitting.